# 02 · Naive baseline

Last-observed value broadcast across the 28-day horizon — the cheapest 
possible baseline. Score is the bottom-level WRMSSE; for the 12-level 
competition score, drop the predictions in `artifacts/cv_naive.parquet` 
and run `06_score.ipynb`.

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import numpy as np
import pandas as pd

from m5.config import SETTINGS, set_global_seed
from m5.data import split_train_horizon
from m5.evaluation import compute_components, make_submission, wrmsse_for_models
from m5.plots import configure_style

configure_style()
set_global_seed()

42

## Build the naive forecast

In [3]:
df = pd.read_parquet(SETTINGS.processed_dir / 'long.parquet')
train, holdout = split_train_horizon(df, horizon=SETTINGS.horizon)

last_value = (train.sort_values(['unique_id', 'ds'])
                  .groupby('unique_id', observed=True)['y']
                  .last())

naive = (holdout[['unique_id', 'ds']]
         .assign(Naive=lambda d: d['unique_id'].map(last_value).astype(np.float32)))
naive.head()

,unique_id,ds,Naive
173,FOODS_1_001_CA_1,2016-04-25,0.0
174,FOODS_1_001_CA_1,2016-04-26,0.0
175,FOODS_1_001_CA_1,2016-04-27,0.0
176,FOODS_1_001_CA_1,2016-04-28,0.0
177,FOODS_1_001_CA_1,2016-04-29,0.0


## Score (bottom-level WRMSSE)

In [4]:
components = compute_components(train)
scores = wrmsse_for_models(holdout[['unique_id', 'ds', 'y']], naive, components, model_cols=['Naive'])
scores

Naive    1.104139
Name: wrmsse, dtype: float64

## Persist for cross-model comparison

In [5]:
out = SETTINGS.artifacts_dir / 'cv_naive.parquet'

cv_long = (holdout[['unique_id', 'ds', 'y']]
           .assign(cutoff=train['ds'].max(), Naive=naive['Naive'].values))
cv_long.to_parquet(out, index=False)
print(f'wrote {out}')

wrote /home/ricka/Git/GitHub/M5/artifacts/cv_naive.parquet


## Optional — Kaggle-format submission

If you want to score with `datasetsforecast.m5.M5Evaluation` (full 12-level 
WRMSSE) or upload to Kaggle, pivot the long forecast into a wide F1..F28 
frame using `m5.evaluation.make_submission`.

In [6]:
submission = make_submission(naive, h=SETTINGS.horizon, value_col='Naive', layout='kaggle')
submission.to_parquet(SETTINGS.forecasts_dir / 'forecast_naive.parquet')
submission.head()

,F1,F2,F3,F4,F5,F6,F7,F8,F9,F10,...,F19,F20,F21,F22,F23,F24,F25,F26,F27,F28
unique_id,,,,,,,,,,,,,,,,,,,,,
FOODS_1_001_CA_1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
FOODS_1_001_TX_2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
FOODS_1_003_CA_4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
FOODS_1_003_TX_3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
FOODS_1_003_WI_1,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,...,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0
